### Dataset Corrections

In [219]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/02_interim/02_dataset_without_leakage.csv', low_memory=False)

#### 1. Grid Position

The dataset saves the exit from pit-lane as zero, what will cause the ML model to learn: "Exit from pit-lane is better than exit from grid position 1". Because of this, the grid position 0 will be changed to max grid pos + 1, based on each race individually, since grid size varies from 1950 to nowadays.

In [220]:
df['grid'] = df['grid'].replace(0, np.nan)

max_grid_per_race = df.groupby('raceId')['grid'].transform('max') + 1

df['grid'] = df['grid'].fillna(max_grid_per_race)
df['grid'] = df['grid'].astype(int)

#### 2. Drop of irrelevant columns for learning
Now, columns without relevant info to the algorithm learn will be dropped. `constructor_url`, for example.

In [221]:
# Constructor Info
df = df.drop(['constructorRef', 'name_constructor', 'nationality_constructor', 'url_constructor'], axis=1)

# Qualifying info
df = df.drop(['constructorId_qualifying', 'number_qualifying'], axis=1)

# Driver Info
df = df.drop(['driverRef', 'number_driver', 'code', 'forename', 'surname', 'dob', 'nationality', 'url_driver'], axis=1)

# Race Info
df = df.drop(['name', 'time_race', 'url'], axis=1)

# FP info
df = df.drop(['fp1_time', 'fp1_date', 'fp2_time', 'fp2_date', 'fp3_time', 'fp3_date'], axis=1)

# Quali and Sprint info
df = df.drop(['quali_time', 'quali_date', 'sprint_time', 'sprint_date'], axis=1)

# Results info
df = df.drop(['number'], axis=1)

df

,statusId,status,qualifyId,raceId,driverId,position_qualifying,q1,q2,q3,constructorId,year,round,circuitId,date,resultId,grid,positionOrder,laps
0,1,Finished,1,18,1,1,1:26.572,1:25.187,1:26.714,1,2008,1,1,2008-03-16,1,1,1,58
1,1,Finished,3,18,5,3,1:25.664,1:25.452,1:27.079,1,2008,1,1,2008-03-16,5,3,5,58
2,1,Finished,5,18,2,5,1:25.960,1:25.518,1:27.236,2,2008,1,1,2008-03-16,2,5,2,58
3,1,Finished,7,18,3,7,1:26.295,1:26.059,1:28.687,3,2008,1,1,2008-03-16,3,7,3,58
4,1,Finished,12,18,4,12,1:26.907,1:26.188,\N,4,2008,1,1,2008-03-16,4,11,4,58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10489,140,Undertray,9411,1084,849,19,1:07.003,\N,\N,3,2022,11,70,2022-07-10,25624,17,19,48
10490,140,Undertray,9420,1085,852,8,1:33.394,1:32.836,1:32.780,213,2022,12,34,2022-07-24,25645,8,20,17
10491,140,Undertray,9881,1111,844,9,1:22.019,1:19.600,1:12.665,6,2023,13,39,2023-08-27,26104,9,19,41
10492,140,Undertray,9989,1116,4,17,1:36.268,\N,\N,117,2023,18,69,2023-10-22,26201,17,16,49


#### 3. Data Typing

In [222]:
df = df.replace(to_replace='\\N', value=np.nan)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10494 entries, 0 to 10493
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   statusId             10494 non-null  int64
 1   status               10494 non-null  str  
 2   qualifyId            10494 non-null  int64
 3   raceId               10494 non-null  int64
 4   driverId             10494 non-null  int64
 5   position_qualifying  10494 non-null  int64
 6   q1                   10338 non-null  str  
 7   q2                   5847 non-null   str  
 8   q3                   3629 non-null   str  
 9   constructorId        10494 non-null  int64
 10  year                 10494 non-null  int64
 11  round                10494 non-null  int64
 12  circuitId            10494 non-null  int64
 13  date                 10494 non-null  str  
 14  resultId             10494 non-null  int64
 15  grid                 10494 non-null  int64
 16  positionOrder        10494 non-nu

Fixing Column Types

In [223]:
df['date'] = pd.to_datetime(df['date'])

q_list = ['q1', 'q2', 'q3']

for q in q_list:
    timedelta_q = pd.to_timedelta('00:0' + df[q])
    target_idx = df.columns.get_loc(q) + 1
    df.insert(target_idx, q + '_millis', timedelta_q // pd.to_timedelta(1, unit='ms'))

In [224]:
df

,statusId,status,qualifyId,raceId,driverId,position_qualifying,q1,q1_millis,q2,q2_millis,...,q3_millis,constructorId,year,round,circuitId,date,resultId,grid,positionOrder,laps
0,1,Finished,1,18,1,1,1:26.572,86572.0,1:25.187,85187.0,...,86714.0,1,2008,1,1,2008-03-16,1,1,1,58
1,1,Finished,3,18,5,3,1:25.664,85664.0,1:25.452,85452.0,...,87079.0,1,2008,1,1,2008-03-16,5,3,5,58
2,1,Finished,5,18,2,5,1:25.960,85960.0,1:25.518,85518.0,...,87236.0,2,2008,1,1,2008-03-16,2,5,2,58
3,1,Finished,7,18,3,7,1:26.295,86295.0,1:26.059,86059.0,...,88687.0,3,2008,1,1,2008-03-16,3,7,3,58
4,1,Finished,12,18,4,12,1:26.907,86907.0,1:26.188,86188.0,...,NaN,4,2008,1,1,2008-03-16,4,11,4,58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10489,140,Undertray,9411,1084,849,19,1:07.003,67003.0,NaN,NaN,...,NaN,3,2022,11,70,2022-07-10,25624,17,19,48
10490,140,Undertray,9420,1085,852,8,1:33.394,93394.0,1:32.836,92836.0,...,92780.0,213,2022,12,34,2022-07-24,25645,8,20,17
10491,140,Undertray,9881,1111,844,9,1:22.019,82019.0,1:19.600,79600.0,...,72665.0,6,2023,13,39,2023-08-27,26104,9,19,41
10492,140,Undertray,9989,1116,4,17,1:36.268,96268.0,NaN,NaN,...,NaN,117,2023,18,69,2023-10-22,26201,17,16,49


#### 4. Temporal Ordering

In [225]:
df = df.sort_values(by=['date', 'raceId'])

df = df.reset_index(drop=True)

#### Saving updated dataset

In [226]:
df.to_csv('../data/02_interim/03_clean_races_info.csv', index=False)